<div style="background-color: white; padding: 10px;">
<center>
    <img style="padding-left:15px"  height='50px' src="https://www.norkart.no/hubfs/norkart-logo-default.svg">
    <img style="padding-left:15px"  height='50px' src="https://www.kartverket.no/public/images/logo/kartverket-logo-large2.svg">
    <br />
    <img style="padding-right:15px" height='50px' src="https://kartai.no/wp-content/uploads/2025/03/cropped-KartAi-med-partnere-2048x1145.png">
    </center>
</div>

# ⛅ Cumulus Geographica - Lær hvordan bruke Cloud Native Geo (CNG)-teknologier? 🗺️

<a target="_blank" href="https://colab.research.google.com/github/kartAI/skygeo/blob/skygeo-workshop/src/skygeo-workshop/Cumulus_Geographica.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

------------
```
TODO
Hvordan unngår du hallusinering? Hvordan kan språkmodeller gjøre GIS-analyser?

Bli med på praktisk workshop der du lærer å kombinere kraften i moderne KI med geografiske data og analyser. I løpet av denne sesjonen vil du:

* Lære hvordan store språkmodeller (LLMs) kan transformere og effektivisere geografiske analyser
* Få hands-on erfaring med å koble ChatGPT-lignende modeller til PostGIS-databaser
* Utforske hvordan du kan stille komplekse geografiske spørsmål på naturlig språk
* Bygge interaktive kart og visualiseringer styrt av AI

Workshopen er designet for både nybegynnere og erfarne geomatikere som ønsker å utforske fremtidens analyseverktøy. Ta med laptop og bli med på å utforske der kunstig intelligens møter geografisk intelligens!

Ingen tidligere KI-erfaring nødvendig – bare ta med din geomatikkunnskap, laptop og god porsjon nysgjerrighet!
```
-------------




# Cumulus Geographica

**???**
* setup - duckdb, s3, httpfs, spatial
    * Forklaring på hva duckdb er??
    * Forklaring på hva geopandas er??
    * Linker til docs for duckdb??

**hvordan jobbe på datasett med milliarder av datapunkter effektivt?**
* Lakehouse-introduksjon - markdown
    * Overture Maps som eksempel
    * Andre åpne kilder? source.coop? Canada?
* Hente filtrerte places og bygninger fra OM - lagre som parquet og som tabeller. Poeng: Vi jobber på datasett med flere milliarder datapunkter og terabyte størrelser. 

**Hva med data som ikke er så store?**
* Bruke geopandas for å laste inn og gjøre analyser mellom kommuner og fylker
    * Hente fra Håvard sitt eksempel
    * Går fint når alt passer i RAM

**Større enn RAM => duckdb**
* Bruke duckdb for å filtrere kun 1 kommune fra parquet-url
* Bruke duckdb for å lage buffer på kommunepolygon. Finne bbox. 
* Filtrere bygninger og places med buffer-bbox med duckdb deretter nøyaktig spatial intersect.

**Refleksjonsspørsmål**
- Hvordan bør norske kartdata organiseres for effektiv bruk?
- Når bør du bruke geopandas? Når bør du bruke duckdb?

**Tips og tricks!**
* Hvordan lage duckdb-sql med agenter og KI?
    * instructions-eksempel



#### ⚙️ Konfigurasjon og oppsett
Kjør cellene under.

In [ ]:
%%capture

%pip install dotenv geopandas folium matplotlib mapclassify duckdb lonboard fsspec adlfs

import os
import geopandas as gpd
import duckdb
from shapely import wkb
from pathlib import Path
from lonboard import Map, PathLayer, viz

kommuner_path = "https://kartaistorage.blob.core.windows.net/skygeo/workshopdata/kommuner_n1000.parquet"
kirkebygg_path = "https://kartaistorage.blob.core.windows.net/skygeo/workshopdata/kirkebygg_forenklet.parquet"


#### ⛅ Hva er Cloud Native Geo-teknologi?

* GeoParquet
* Hive-partitioning `*` er magien!
* OvertureMaps som eksempel
* Linker til mer ressurser

In [ ]:
# setup duckdb connection
# Initialize DuckDB connection
conn = duckdb.connect(':memory:')

#install spatial
conn.execute("INSTALL spatial;")

# Load spatial extension
conn.execute("LOAD spatial;")

# Set S3 region for Overture Maps
conn.execute("SET s3_region='us-west-2';")

#### Hvordan jobbe på datasett med milliarder av datapunkter effektivt?

Nå skal du prøve ut datauthenting fra OvertureMaps

1. Kjør cellen under
1. Se på dataschemaet ... http://....
- https://docs.overturemaps.org/theme-definitions-table.html
- https://docs.overturemaps.org/guides/places/
- https://github.com/OvertureMaps/schema/blob/main/docs/schema/concepts/by-theme/places/overture_categories.csv


1. Kan du hente ut kun ....


In [ ]:
bbox_kristiansand = [7.875366,58.081425,8.099899,58.206042]
bbox = bbox_kristiansand
# Query pizza restaurants in Kristiansand from Overture Maps

query = f"""
SELECT
    id,
    names.primary as name,
    confidence,
    categories.primary as category,
    CAST(categories as JSON) as categories,
    CAST(socials AS JSON) as socials,
    geometry
FROM
    read_parquet('s3://overturemaps-us-west-2/release/2026-02-18.0/theme=places/type=place/*', 
                 filename=true, hive_partitioning=1)
WHERE
    categories.primary IN ('pizza_restaurant','pizza_delivery_service')
    AND bbox.xmin BETWEEN {bbox[0]} AND {bbox[2]}
    AND bbox.ymin BETWEEN {bbox[1]} AND {bbox[3]}
"""

# Execute query
result = conn.sql(query)

display(result.df())


#### Lag et interaktivt kart av resultatet

In [ ]:
viz(result)

### Lagre spørringen som en tabell i DuckDB

In [ ]:

conn.sql(f"""
CREATE OR REPLACE TABLE overturemap_results AS
    {query}
""")

conn.sql("SELECT * FROM overturemap_results").df().head(5)

In [ ]:
# Verify the geometry column exists and its type
r = conn.sql("DESCRIBE overturemap_results")
r.df()
# Or query just the column info
#conn.sql("SELECT column_name, data_type FROM information_schema.columns WHERE table_name='pizza_restaurants'").show()

### Lagre resultatet av en spørring som en Parquet fil på disk

In [ ]:
# export the duckdb result to a parquet file and read it with geopandas
tmp_parquet_path = "./tmp/duckdb_result.parquet"
result.to_parquet(tmp_parquet_path)

### Bruke GeoPandas på resultatet
- Lese inn parquet-fil (fra lokal disk) direkte med GeoPandas
- .plot() = "matematisk" plotting
- .explore() = interaktivt kart

In [ ]:
# read the parquet file with geopandas
gdf = gpd.read_parquet(tmp_parquet_path)
gdf.head()

In [ ]:
gdf.plot()
gdf.explore()

### Bruke GeoPandas for analyser

- Håvard-inspirasjon

In [ ]:
kommune_uri = "az://skygeo/workshopdata/kommuner_n1000.parquet"
account_name = "kartaistorage"

kommune_original = gpd.read_parquet(
    kommune_uri, 
    storage_options={"account_name": account_name}
)

kirkebygg_uri = "az://skygeo/workshopdata/kirkebygg_forenklet.parquet"

kirkebygg = gpd.read_parquet(
    kirkebygg_uri, 
    storage_options={"account_name": account_name}
)

**..**

In [ ]:
kommune_original = kommune_original[kommune_original["kommunenummer"] == "3305"][["kommunenummer", "kommunenavn", "geometry"]].copy()

kommune_buffer = kommune_original.copy()

kommune_buffer = kommune_buffer.to_crs(3857)

kommune_buffer["geometry"] = kommune_buffer.buffer(15_000) # REFLEKSJON: Hva gjør vi her?

kommune_buffer = kommune_buffer.to_crs(4326)

**..**

In [ ]:


kirker_i_original = kirkebygg[["bygningsnavn", "godkjentkapasitet", "hovedmateriale", "geometry"]].clip(kommune_original)

kirker_i_original_antall = len(kirker_i_original)

kirker_i_original_kapasitet = kirker_i_original["godkjentkapasitet"].sum()

display(f"Antall kirker i kommune er {kirker_i_original_antall} og rommer {kirker_i_original_kapasitet} personer.")

kirker_i_buffer = kirkebygg[["bygningsnavn", "godkjentkapasitet", "hovedmateriale", "geometry"]].clip(kommune_buffer)

kirker_i_buffer_antall = len(kirker_i_buffer)

kirker_i_buffer_kapasitet = kirker_i_buffer["godkjentkapasitet"].sum()

display(f"Antall kirker i buffer er {kirker_i_buffer_antall} og rommer {kirker_i_buffer_kapasitet} personer.")

**Visualiser resultatene**

In [ ]:
import folium

In [ ]:
xmin, ymin, xmax, ymax = kommune_buffer.total_bounds.tolist()

m = folium.Map(
    tiles="cartodb positron",
)

m.fit_bounds([[ymin, xmin], [ymax, xmax]])

folium.GeoJson(
    data=kommune_original,
    color="blue",
    fill_opacity=0.0,
    weight=2.0,
).add_to(m)

folium.GeoJson(
    data=kommune_buffer,
    color="red",
    fill_opacity=0.0,
    weight=1.0,
    dash_array=[10.0, 5.0],
).add_to(m)

folium.GeoJson(
    data=kirker_i_buffer,
    tooltip=folium.GeoJsonTooltip(fields=["bygningsnavn"]),
    marker=folium.CircleMarker(
        color="black",
        fill_color="white",
        fill_opacity=1.0,
        radius=5.0,
        weight=1,
    ),
).add_to(m)

m

### Hva med analyser på store datasett?

- ...

In [ ]:
# Kommune 3305: transformer til EPSG:25833, buffer 15000 meter, transformer tilbake for intersect
os.makedirs('./tmp', exist_ok=True)

conn.sql(f"""
    CREATE OR REPLACE TABLE kirkebygg_i_kommune_3305 AS
    WITH kommune_buffer AS (
        SELECT
            ST_Transform(
                ST_Buffer(
                    ST_Transform(geometry, 'EPSG:4326', 'EPSG:25833'),
                    15000
                ),
                'EPSG:25833',
                'EPSG:4326'
            ) AS geometry
        FROM read_parquet('{kommuner_path}')
        WHERE CAST(kommunenummer AS VARCHAR) = '3305'
        LIMIT 1
    )
    SELECT k.*
    FROM read_parquet('{kirkebygg_path}') AS k
    JOIN kommune_buffer AS b
      ON ST_Intersects(k.geometry, b.geometry)
""")

# lagre resultater som parquet
conn.sql("COPY kirkebygg_i_kommune_3305 TO './tmp/kirkebygg_i_kommune_3305.parquet' (FORMAT PARQUET)")

conn.sql("SELECT COUNT(*) AS antall_kirkebygg FROM kirkebygg_i_kommune_3305").df()

In [ ]:
# Stegvis for å tvinge tidlig filtrering (predicate pushdown)
os.makedirs('./tmp', exist_ok=True)

# 1) Hent kommune 3305-geometri som egen tabell
conn.sql(f"""
    CREATE OR REPLACE TABLE kommune_3305 AS
    SELECT geometry
    FROM read_parquet('{kommuner_path}')
    WHERE CAST(kommunenummer AS VARCHAR) = '3305'
    LIMIT 1
""")

print("Kommune 3305 geometri lastet inn i DuckDB.")

# 2) Hent bbox-verdier til Python (literals i neste query)
xmin, ymin, xmax, ymax = conn.sql("""
    SELECT
        ST_XMin(geometry) AS xmin,
        ST_YMin(geometry) AS ymin,
        ST_XMax(geometry) AS xmax,
        ST_YMax(geometry) AS ymax
    FROM kommune_3305
""").fetchone()

print(f"BBOX for kommune 3305: xmin={xmin}, ymin={ymin}, xmax={xmax}, ymax={ymax}")

# 3) Kun bbox-filter mot Overture (remote), og lagre lokalt parquet
conn.sql(f"""
    COPY (
        SELECT
            id,
            CAST(sources AS JSON) AS sources,
            subtype,
            class,
            geometry
        FROM read_parquet(
            's3://overturemaps-us-west-2/release/2026-02-18.0/theme=buildings/type=building/*',
            filename=true,
            hive_partitioning=1
        )
        WHERE
            bbox.xmax >= {xmin}
            AND bbox.xmin <= {xmax}
            AND bbox.ymax >= {ymin}
            AND bbox.ymin <= {ymax}
    ) TO './tmp/overture_buildings_bbox_3305.parquet' (FORMAT PARQUET)
""")

print("Bygningsdata for bbox lastet ned og lagret lokalt.")

# 4) Presis spatial intersect på den lokale, forhåndsfiltrerte filen
conn.sql("""
    CREATE OR REPLACE TABLE overture_buildings_i_kommune_3305 AS
    SELECT b.*
    FROM read_parquet('./tmp/overture_buildings_bbox_3305.parquet') b
    CROSS JOIN kommune_3305 k
    WHERE ST_Intersects(b.geometry, k.geometry)
""")

print("Presis spatial intersect utført på lokal fil.")

conn.sql("SELECT COUNT(*) AS antall_bygninger FROM overture_buildings_i_kommune_3305").df()

In [ ]:
conn.sql("SELECT * FROM overture_buildings_i_kommune_3305").df()

### Finn alle Bygninger som _ikke_ er boligbygg (residential) og har over 10.000kvm areal innenfor kommunen og buffersonen

In [ ]:
res = conn.sql("""
    SELECT 
        *,
        st_area(st_transform(geometry,'EPSG:4326','EPSG:25833')) as area,
        st_astext(geometry) geometry_wkt
    FROM overture_buildings_i_kommune_3305
    WHERE
        subtype NOT IN ('residential')
        AND area > 10000
         """)
display(res.df())

viz(res)

### Oppgave: Finn alle Places i kommunen!

### Ekstraoppgave: Finn alle Places innenfor 100 meter fra alle Kirker

### Eksportere resultater

In [ ]:
#Export csv, tabell og plotkart + html-kart til disk